In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
def split_address(address):
    parts = address.split()
    
    if len(parts) >= 4:
        city_name = parts[0]  # 예: 전라북도
        district = parts[1]  # 예: 남원시
        town = parts[2]  # 예: 대강면
        road_name = ' '.join(parts[3:-1])  # 예: 대강월산길 (주소의 마지막 요소 전까지 모두 도로명으로 처리)
        bungee = parts[-1]  # 예: 37-16 (주소의 마지막 요소는 항상 번지로 처리)
        
        return city_name, district, town, road_name, bungee
    else:
        return None, None, None, None, None

In [ ]:
df1 = pd.read_csv('../data/namwon2/전라북도 남원시 축산현황 및 가금류 농가현황_20230727.csv', encoding='cp949')
df2 = pd.read_csv('../data/namwon2/전라북도 남원시_태양광발전소 인허가 및 설치현황_20231005.csv', encoding='cp949')

df3_1 = pd.read_csv('../data/namwon2/전라북도 남원시_5층이상대형병원현황_20240423.csv', encoding='cp949')
df3_2 = pd.read_csv('../data/namwon2/전북특별자치도 남원시_정신건강관련시설_20240711.csv', encoding='cp949')
df3_3 = pd.read_csv('../data/namwon2/전라북도 남원시_동물병원현황_20231006.csv', encoding='cp949')

In [ ]:
df1['num'] = 1

In [ ]:
fig = px.bar(df1, x='행정동명', y='num')
fig.show()

In [ ]:
df2

In [ ]:
df2['num'] = 1
df2[['시도', '시군구', '읍면동', '도로명', '번지']] = df2['주소'].apply(lambda x: pd.Series(split_address(x)))

In [ ]:
df2

In [ ]:
fig = px.bar(df2, x='읍면동', y='num')
fig.show()

In [ ]:
df3_1['num'] = 1
df3_2['num'] = 1
df3_3['num'] = 1

df3_1[['시도', '시군구', '읍면동', '도로명', '번지']] = df3_1['도로명 소재지'].apply(lambda x: pd.Series(split_address(x)))
df3_2[['시도', '시군구', '읍면동', '도로명', '번지']] = df3_2['도로명주소'].apply(lambda x: pd.Series(split_address(x)))
df3_3[['시도', '시군구', '읍면동', '도로명', '번지']] = df3_3['도로명주소'].apply(lambda x: pd.Series(split_address(x)))

In [ ]:
df3_2['구분'] = '정신병원'
df3_3['구분'] = '동물병원'

In [ ]:
df3_1 = df3_1.loc[:, ['도로명 소재지', '읍면동', '의료기관명', '종류', 'num']].rename(columns={'도로명 소재지': '소재지', '의료기관명': '시설명', '종류': '구분'})
df3_2 = df3_2.loc[:, ['도로명주소', '읍면동', '업체명', '구분', 'num']].rename(columns={'도로명주소': '소재지', '업체명': '시설명'})
df3_3 = df3_3.loc[:, ['도로명주소', '읍면동', '동물병원명', '구분', 'num']].rename(columns={'도로명주소': '소재지', '동물병원명': '시설명'})

In [ ]:
df1 = df1.loc[:, ['주소지', '행정동명', '농장명', '축종', 'num']].rename(columns={'주소지': '소재지', '행정동명': '읍면동', '농장명': '시설명', '축종': '구분'})
df2 = df2.loc[:, ['주소', '읍면동', '발전소명', 'num']].rename(columns={'주소': '소재지', '발전소명': '시설명'})
df3 = pd.concat([df3_1, df3_2, df3_3]).reset_index().drop('index', axis=1)

In [ ]:
df2

In [ ]:
import duckdb

In [ ]:
conn = duckdb.connect('../data/database.db')

In [ ]:
conn.execute('CREATE TABLE farm AS SELECT * FROM df1')
conn.execute('CREATE TABLE solar_power_plant AS SELECT * FROM df2')
conn.execute('CREATE TABLE hospital AS SELECT * FROM df3')

In [ ]:
conn.close()